# Check the cleaned tabular data

This notebook compares the raw `ListedCompanyFeatures772_<year>.csv` files with the output of `src/clean_tabular.py`.
Each section shows a before/after example and ends with a check that prints `PASS` or raises an error.

Run `python src/clean_tabular.py` first so that `data/processed/` exists.

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30, "display.width", 200)

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
KEYS = ["nodeID", "Year", "Label"]

raw = pd.concat(
    [pd.read_csv(ROOT / "data" / str(y) / f"ListedCompanyFeatures772_{y}.csv", low_memory=False)
     for y in range(2014, 2023)],
    ignore_index=True,
).sort_values(["Year", "nodeID"], ignore_index=True)
clean = pd.read_parquet(ROOT / "data" / "processed" / "tabular_clean")
clean = clean.sort_values(["Year", "nodeID"], ignore_index=True)
log = pd.read_csv(ROOT / "data" / "processed" / "cleaning_log.csv")

print("raw  :", raw.shape)
print("clean:", clean.shape)

raw  : (33664, 775)
clean: (33664, 669)


## 1. No rows were lost, added or relabelled

Cleaning should only change feature values and columns. The company-years and their labels must be identical.

In [2]:
assert raw[KEYS].equals(clean[KEYS].astype(raw[KEYS].dtypes)), "rows or labels differ"
assert not clean.duplicated(["nodeID", "Year"]).any()

summary = pd.DataFrame({
    "raw rows": raw.groupby("Year").size(),
    "clean rows": clean.groupby("Year").size(),
    "raw fraud": raw.groupby("Year")["Label"].sum(),
    "clean fraud": clean.groupby("Year")["Label"].sum(),
})
print("PASS: same company-years and labels")
summary

PASS: same company-years and labels


,raw rows,clean rows,raw fraud,clean fraud
Year,,,,
2014,2631,2631,95,95
2015,2824,2824,134,134
2016,3125,3125,173,173
2017,3503,3503,205,205
2018,3596,3596,225,225
2019,3815,3815,214,214
2020,4269,4269,207,207
2021,4769,4769,183,183
2022,5132,5132,123,123


## 2. Columns: 772 features in, 666 out

Every raw column appears in the cleaning log with what happened to it.

In [3]:
kept = log.loc[log["action"] == "kept", "column"].tolist()
assert len(log) == 772
assert list(clean.columns) == KEYS + kept, "cleaned columns do not match the log"

print("PASS: cleaned table has exactly the columns the log marks as kept")
log["action"].str.replace(r"identical to .*", "identical to another column", regex=True).value_counts()

PASS: cleaned table has exactly the columns the log marks as kept


action
kept                                    666
dropped: identical to another column     99
dropped: constant                         7
Name: count, dtype: int64

## 3. Blank statement items became 0

The problem in the raw data: for financial statement line items, the 2014-2015 files write an unreported item as `0`, while later files leave it blank.

**Example.** `A0d1102000` (settlement reserve funds) only applies to financial firms, so nearly every company has nothing to report. In the raw data the "nothing" switches from zero to blank in 2016, while the share of companies with a real value barely changes.

In [4]:
def value_mix(df, col):
    s = df[col]
    kind = pd.Series("non-zero", index=s.index).mask(s == 0, "zero").mask(s.isna(), "blank")
    return pd.crosstab(df["Year"], kind, normalize="index").round(3)

example = "A0d1102000"
pd.concat({"raw": value_mix(raw, example), "clean": value_mix(clean, example)}, axis=1).fillna(0)

raw                    clean       
col_0  blank non-zero   zero non-zero   zero
Year                                        
2014   0.003    0.010  0.987    0.010  0.990
2015   0.006    0.011  0.983    0.011  0.989
2016   0.017    0.013  0.969    0.013  0.987
2017   0.986    0.014  0.000    0.014  0.986
2018   0.985    0.014  0.001    0.014  0.986
2019   0.983    0.015  0.002    0.015  0.985
2020   0.985    0.015  0.001    0.015  0.985
2021   0.986    0.013  0.001    0.013  0.987
2022   0.984    0.013  0.003    0.013  0.987

The same company before and after. `L000002` in 2015 (old style) and 2017 (new style), for a few balance sheet items:

In [5]:
cols = ["A001101000", "A0d1102000", "A0b1103000", "A001107000", "A001111000"]
rows = lambda df: df[(df["nodeID"] == "L000002") & (df["Year"].isin([2015, 2017]))].set_index("Year")[cols]
pd.concat({"raw": rows(raw), "clean": rows(clean)}).rename(
    columns=log.set_index("column")["description"].str.extract(r"\[(.*?)\]")[0]
)

Monetary Funds  Settlement Reserve Funds  Cash and Deposits with the Central Bank  Trading Financial Assets  Net Accounts Receivable
      Year                                                                                                                                      
raw   2015    5.318038e+10                       0.0                                      0.0                       0.0             2.510653e+09
      2017    1.741210e+11                       NaN                                      NaN                       NaN             1.432734e+09
clean 2015    5.318038e+10                       0.0                                      0.0                       0.0             2.510653e+09
      2017    1.741210e+11                       0.0                                      0.0                       0.0             1.432734e+09

Missing cells per year, before and after. The jump in 2016 disappears.

In [6]:
features = [c for c in raw.columns if c not in KEYS]
pd.DataFrame({
    "raw % blank": raw[features].isna().mean(axis=1).groupby(raw["Year"]).mean() * 100,
    "clean % blank": clean[kept].isna().mean(axis=1).groupby(clean["Year"]).mean() * 100,
}).round(1)

,raw % blank,clean % blank
Year,,
2014,5.6,2.1
2015,5.2,2.4
2016,23.1,2.6
2017,26.1,2.3
2018,25.5,2.3
2019,25.3,2.3
2020,25.5,2.6
2021,25.2,2.7
2022,25.3,2.8


Three things must hold:

1. A value that existed in the raw data is never altered.
2. The only change is blank to 0, and only in statement columns (codes starting A, B, C or D).
3. Ratio and other columns are untouched, blanks included.

In [7]:
is_statement = lambda c: re.fullmatch(r"[A-D][0-9A-Za-z]{2}\d{7}", c) is not None
numeric_kept = [c for c in kept if c != "IndustryCodeC"]
statement = [c for c in numeric_kept if is_statement(c)]
other = [c for c in numeric_kept if not is_statement(c)]

r, c = raw[numeric_kept], clean[numeric_kept]
had_value = r.notna()
assert (r.where(had_value) == c.where(had_value)).where(had_value, True).all().all(), "an existing value changed"

filled = r.isna() & c.notna()
assert (c.where(filled, 0) == 0).all().all(), "a blank was filled with something other than 0"
assert not filled[other].any().any(), "a non-statement column was filled"
assert raw[other].equals(clean[other]), "a non-statement column changed"

print("PASS: existing values unchanged")
print(f"PASS: {int(filled.sum().sum()):,} blanks set to 0, all in the {len(statement)} statement columns")
print(f"PASS: the {len(other)} ratio/other columns are identical to the raw data")
print(f"blanks still left in statement columns: {int(c[statement].isna().sum().sum())} "
      "(companies with no values at all for that statement)")

PASS: existing values unchanged
PASS: 4,857,999 blanks set to 0, all in the 294 statement columns
PASS: the 371 ratio/other columns are identical to the raw data
blanks still left in statement columns: 52 (companies with no values at all for that statement)


## 4. Dropped columns were constant or duplicates

**Constant columns** have at most one distinct value, so they cannot separate fraud from non-fraud.

In [8]:
constant = log.loc[log["action"] == "dropped: constant", "column"]
table = pd.DataFrame({
    "distinct values": [raw[c].nunique() for c in constant],
    "the value": [raw[c].dropna().unique().tolist() for c in constant],
    "description": log.set_index("column").loc[constant, "description"].str.extract(r"\[(.*?)\]")[0].tolist(),
}, index=constant)
assert (table["distinct values"] <= 1).all()
print("PASS: all 7 have at most one distinct value")
table

PASS: all 7 have at most one distinct value


,distinct values,the value,description
column,,,
A001109000,1,[0.0],Net Short-Term Investments
A001206000,1,[0.0],Net Long-Term Debt Investments
A0d2101101,1,[0.0],Among them: Pledged Borrowings
A003106000,1,[0.0],Foreign Currency Translation Differences
A003107000,1,[0.0],Add: Unrecognized Investment Losses
B002200000,1,[0.0],Unrecognized Investment Losses
D000117000,1,[0.0],Unrecognized Investment Losses


**Duplicate columns** are identical, row for row, to a column that was kept. Example: three versions of the same ratio.

In [9]:
trio = ["F040203B", "F040204B", "F040205C"]
print(log.set_index("column").loc[trio, "description"].str.slice(0, 60).to_string(), "\n")
raw.loc[raw["Year"] == 2020, ["nodeID"] + trio].head(6)

column
F040203B    [Accounts Receivable Turnover Rate C] - Calcul...
F040204B    [Accounts Receivable Turnover Rate D] - Calcul...
F040205C    [Accounts Receivable Turnover Rate TTM] - Calc... 



,nodeID,F040203B,F040204B,F040205C
19494,L000001,NaN,NaN,NaN
19495,L000002,168.301078,168.301078,168.301078
19496,L000004,0.968174,0.968174,0.968174
19497,L000005,1.088005,1.088005,1.088005
19498,L000006,192.651549,192.651549,192.651549
19499,L000007,NaN,NaN,NaN


In [10]:
dups = log[log["action"].str.startswith("dropped: identical")].copy()
dups["match"] = dups["action"].str.replace("dropped: identical to ", "")

# compare after the blank -> 0 step, since that is when duplicates were detected
harmonised = raw.copy()
for prefix in "ABCD":
    block = [c for c in features if is_statement(c) and c[0] == prefix]
    has_any = harmonised[block].notna().any(axis=1)
    harmonised.loc[has_any, block] = harmonised.loc[has_any, block].fillna(0.0)

same = [harmonised[a].equals(harmonised[b]) for a, b in zip(dups["column"], dups["match"])]
assert all(same), "a dropped column is not identical to its match"
assert dups["match"].isin(kept).all(), "a match column was itself dropped"
print(f"PASS: all {len(dups)} dropped duplicates equal a kept column in all {len(raw):,} rows")

PASS: all 99 dropped duplicates equal a kept column in all 33,664 rows


## 5. Industry code

Twelve company-years had no industry code. Each was filled from the same company's nearest year.

In [10]:
blank = raw["IndustryCodeC"].isna()
filled_rows = clean.loc[blank, ["nodeID", "Year", "IndustryCodeC"]].rename(columns={"IndustryCodeC": "filled with"})
filled_rows["company's codes in the raw data"] = [
    sorted(raw.loc[raw["nodeID"] == n, "IndustryCodeC"].dropna().unique()) for n in filled_rows["nodeID"]
]
assert clean["IndustryCodeC"].notna().all()
assert all(v in codes for v, codes in zip(filled_rows["filled with"], filled_rows["company's codes in the raw data"]))
assert raw.loc[~blank, "IndustryCodeC"].equals(clean.loc[~blank, "IndustryCodeC"])
print("PASS: no blanks left, each fill comes from the same company, existing codes unchanged")
filled_rows

PASS: no blanks left, each fill comes from the same company, existing codes unchanged


,nodeID,Year,filled with,company's codes in the raw data
2605,L603519,2014,C38,[C38]
5454,L832110,2015,C39,[C39]
11558,L601360,2017,I64,[I64]
15668,L831855,2018,C34,[C34]
17884,L430425,2019,I65,[I65]
19476,L831855,2019,C34,[C34]
19482,L833394,2019,C17,[C17]
21899,L430425,2020,I65,[I65]
23746,L831855,2020,C34,[C34]
23750,L833394,2020,C17,[C17]


## 6. What is still blank

These are left for the feature-engineering step, because imputing them needs statistics from the training folds only.

In [11]:
left = log[(log["action"] == "kept") & (log["missing_clean"] > 0)]
print(f"{len(left)} of {len(kept)} kept columns still have blanks; overall {clean[kept].isna().mean().mean():.1%} of cells")
left.sort_values("missing_clean", ascending=False).head(10)[["column", "group", "missing_clean", "description"]]

346 of 666 kept columns still have blanks; overall 2.5% of cells


,column,group,missing_clean,description
577,F082301B,ratio / other,0.3391,[Net Cash Flow from Operating Activities Growt...
575,F082201B,ratio / other,0.3391,[Per Share Net Cash Flow from Operating Activi...
716,F110501B,ratio / other,0.3250,[Dividend Change Rate per Share] - Calculation...
310,F010701B,ratio / other,0.3215,[Interest Coverage Ratio A] - Calculation form...
313,F010901B,ratio / other,0.3215,[Cash Flow Interest Coverage Ratio] - Calculat...
311,F010702B,ratio / other,0.3215,[Interest Coverage Ratio B] - Calculation form...
707,F101101B,ratio / other,0.3095,[Price to Dividend Ratio] - Calculation formul...
712,F110101B,ratio / other,0.2862,[Pre-tax Cash Dividend per Share] - The pre-ta...
713,F110201B,ratio / other,0.2862,[Post-tax Cash Dividend per Share] - The post-...
717,F110601B,ratio / other,0.2862,[Cash Dividend Coverage Ratio] - Calculation f...


In [17]:
import pandas as pd

df = pd.read_parquet("../data/processed/tabular_clean")

#df.head(20)
#df[df["Year"] == 2020]                 # one year
#df[df["nodeID"] == "L000002"]          # one company across all years
#df[df["Label"] == 1]                   # fraud cases only
df.describe().T                        # summary stats for every feature


,count,mean,std,min,25%,50%,75%,max
Year,33664.0,2.018548e+03,2.552594e+00,2014.000000,2.016000e+03,2.019000e+03,2.021000e+03,2.022000e+03
Label,33664.0,4.631060e-02,2.101600e-01,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
RegisterLongitude,33637.0,1.158910e+02,6.353710e+00,76.037106,1.135683e+02,1.167571e+02,1.203746e+02,1.329817e+02
RegisterLatitude,33637.0,3.143654e+01,6.004676e+00,18.222090,2.815661e+01,3.115165e+01,3.609050e+01,4.785380e+01
A001101000,33664.0,2.949510e+09,1.491047e+10,0.000000,2.387918e+08,5.635404e+08,1.456126e+09,6.125950e+11
...,...,...,...,...,...,...,...,...
ListingAge,26749.0,1.201002e+01,7.588507e+00,0.000000,5.000000e+00,1.000000e+01,1.900000e+01,3.200000e+01
AssetSize,26749.0,2.237914e+01,1.350294e+00,14.940000,2.147000e+01,2.221000e+01,2.312000e+01,2.864000e+01
StockYield,26744.0,1.233971e-01,5.531074e-01,-0.923704,-2.225992e-01,-6.365000e-04,3.094115e-01,1.428014e+01
InefficInvestDegree,26749.0,4.819157e-02,1.745863e-01,0.000000,1.103600e-02,2.510400e-02,5.077100e-02,1.755158e+01
